In [1]:
import re
import polars as pl
import pykakasi
from pathlib import Path

KAKASI_INST = pykakasi.kakasi()

KKS_PATTERNS = [
    (re.compile(r'角樽', re.I), 'かくたる'),
    (re.compile(r'取っ手', re.I), 'とって'),
    (re.compile(r'取手', re.I), 'とって'),
    (re.compile(r'型', re.I), 'がた'),
    (re.compile(r'BOX', re.I), 'ぼっくす'),
    (re.compile(r'PET', re.I), 'ぺっと'),
    (re.compile(r'MF', re.I), 'えむえふ'),
    (re.compile(r'蓋', re.I), 'ふた'),
    (re.compile(r'中蓋', re.I), 'なかふた'),
    (re.compile(r'#', re.I), 'しゃーぷ'),
    (re.compile(r'長側', re.I), 'ちょうそく'),
    (re.compile(r'短側', re.I), 'たんそく'),
]

ZEN = "".join(chr(0xff01 + idx) for idx in range(94))
HAN = "".join(chr(0x21 + idx) for idx in range(94))
ZEN2HAN_TABLE = str.maketrans(ZEN, HAN)
CLEAN_TABLE = str.maketrans({'\n': '', ' ': '', '　': '', '-': '', '.': '', '•': '', '・': '', '/': '', '／': ''})

# 各種正規表現も事前コンパイル
RE_CYCLE = re.compile(r'\d+(?:\.\d+)?')
RE_MFR_KEY = re.compile(r'^[A-Za-z]+-\d+$')
RE_CHART_FILE = re.compile(r'山積み.*\d{8}\.xlsx')
RE_CHART_DATE = re.compile(r'\d{8}')
RE_MFR_FILE = re.compile(r'MFR\.xlsx')
RE_NEWLINE = re.compile(r'[\r\n]+')
# =============================================================================

def kks(goods, resource, sk, i_2, i_4):
    trans = goods
    for pattern, replacement in KKS_PATTERNS:
        trans = pattern.sub(replacement, trans)
    
    romaji = KAKASI_INST.convert(trans)
    
    # リスト内包表記を使用して文字列結合処理を最適化
    hira = ''.join([item['hira'] for item in romaji])
    kana = ''.join([item['kana'] for item in romaji])
    hepburn = ''.join([item['hepburn'] for item in romaji])
    kunrei = ''.join([item['kunrei'] for item in romaji])
    passport = ''.join([item['passport'] for item in romaji])
    
    keyword = f'{sk},{i_2},{i_4},{hira},{kana},{hepburn},{kunrei},{passport}'
    keyword = keyword.translate(CLEAN_TABLE)
    keyword = keyword.translate(ZEN2HAN_TABLE)
    
    if resource == '未登録':
        keyword += ',未登録'
    return keyword

def is_numeric(value):
    try:
        float(value)
        return True
    except (ValueError, TypeError):
        return False

def format_number(num):
    rounded = round(num, 2)
    # 文字列のフォーマット処理をシンプルにして高速化
    return f"{rounded:.2f}".rstrip('0').rstrip('.')

def resources(stack_chart, cycle, tori, s_cycle):
    if not isinstance(stack_chart, dict):
        return {'finish': '未登録', 'unit': '未登録', 'resource': '未登録', 'abnormal': '未登録'}
    
    c = RE_CYCLE.search(str(s_cycle))
    s_cycle_val = c.group() if c else 0
    
    try:
        cycle_val = float(cycle) if is_numeric(cycle) else float(s_cycle_val)
        finish = float(stack_chart.get(5, 0)) if is_numeric(stack_chart.get(5)) else 0
        unit = float(stack_chart.get(6, 0)) if is_numeric(stack_chart.get(6)) else 0
        time_val = float(stack_chart.get(7, 0)) if is_numeric(stack_chart.get(7)) else 0
        walk = float(stack_chart.get(8, 0)) if is_numeric(stack_chart.get(8)) else 0
        abnormal = float(stack_chart.get(9, 0)) if is_numeric(stack_chart.get(9)) else 0
        
        result = (3600 / cycle_val) * float(tori)
        result = result / unit
        result = (result * finish) / 3600
        result = round(result, 2)
        
        return {
            'finish': format_number(finish), 
            'unit': format_number(unit), 
            'resource': format_number(result), 
            'abnormal': format_number(abnormal)
        }
    except Exception:
        return {'finish': '不詳', 'unit': '不詳', 'resource': '不詳', 'abnormal': '不詳'}

def read_excel_to_dict_list(filepath, sheet_name=None):
    if not filepath.exists():
        return []
        
    if sheet_name:
        df = pl.read_excel(filepath, sheet_name=sheet_name, has_header=False, engine="calamine")
    else:
        df = pl.read_excel(filepath, has_header=False, engine="calamine")
    
    df = df.cast(pl.String).fill_null("")
    dict_list = []
    
    # DataFrameの行イテレーションを最適化
    # iter_rows() はタプルを返すため rows() よりメモリ効率が高く高速です
    for row in df.iter_rows():
        # 辞書の初期化を高速化（内包表記で1〜24のキーを事前作成）
        d = {j: "" for j in range(1, 25)}
        for j, val in enumerate(row[1:]):
            if val.endswith('.0'):
                check_val = val[:-2].lstrip('-')
                if check_val.isdigit():
                    val = val[:-2]
            d[j+1] = val.strip()
        dict_list.append(d)
        
    return dict_list

def main():
    base_dir = Path('./cabinet/')
    file_path = base_dir / 'data.xlsx'
    data = read_excel_to_dict_list(file_path, sheet_name='データ')[1:]

    # iterdir() とコンパイル済み正規表現で効率的にファイルをフィルタリング
    chart_files = [f for f in base_dir.iterdir() if f.is_file() and RE_CHART_FILE.search(f.name)]
    latest_file = max(chart_files, key=lambda x: RE_CHART_DATE.search(x.name).group(), default=None)

    if latest_file:
        chart = read_excel_to_dict_list(latest_file)
    else:
        chart = []
    
    # 辞書内包表記で効率化
    chart_dict = {str(d.get(2)) + '-' + str(d.get(3)): d for d in chart if 2 in d}

    mfr_files = [f for f in base_dir.iterdir() if f.is_file() and RE_MFR_FILE.search(f.name)]
    mfr = read_excel_to_dict_list(mfr_files[0]) if mfr_files else []

    mfr_dict = {}
    for d in mfr:
        key_col = None
        for j in range(1, 6):
            if RE_MFR_KEY.match(str(d.get(j, '')).strip()):
                key_col = j
                break
        
        if key_col:
            key = str(d.get(key_col)).replace('-', '').strip()
            shift = key_col - 2
            d['raw_val'] = d.get(5 + shift, "")
            d['raw_mfr_val'] = d.get(6 + shift, "")
            mfr_dict[key] = d

    products = []
    for i in data:
        tori = 1 if i.get(13) == '' else int(i.get(13))
        hour = 0 if i.get(12) == '' else (3600 / float(i.get(12))) * tori
        keys = [item if '-' in item else f"{item}-00" for item in str(i.get(2, '')).split()]
        chart_result = next((chart_dict[key] for key in keys if key in chart_dict), None)
        stack = resources(chart_result, i.get(12, ''), tori, i.get(6, ''))
        
        sk_n = str(i.get(1, '')).replace('.', '')
        sk = f'SK{sk_n}' if i.get(7) == '' else f'{str(i.get(7)).upper()}{sk_n}'
        
        code = str(i.get(2, '')).replace('\n', ' ')
        name = str(i.get(4, '')).replace('\n', '・')
        time_val = stack["finish"]
        unit = stack["unit"]
        skill = stack["resource"]
        abnormal = stack["abnormal"]
        
        grossWeight = f'{i.get(18)}{"ｇ" if i.get(18) != "" else ""}'
        wgt = str(i.get(17, '')).replace('±', ' ±')
        spawn = str(i.get(13, ''))
        cycle_val = str(i.get(12, ''))
        standard = i.get(6) if i.get(6) != "" else ""
        material = str(i.get(11, ''))
        
        mfr_entry = mfr_dict.get(sk, {})
        raw = RE_NEWLINE.sub(' / ', str(mfr_entry.get('raw_val', "")))
        raw_mfr = RE_NEWLINE.sub(' / ', str(mfr_entry.get('raw_mfr_val', "")))
        
        one_box = str(i.get(14, ''))
        pallet = str(i.get(15, ''))
        tape = str(i.get(21, ''))
        box = str(i.get(22, ''))
        bag = str(i.get(23, ''))
        
        # タプルを用いたリスト内包表記へ簡略化
        _etc = [str(i.get(k, "")).strip() for k in (19, 20) if i.get(k)]
        etc = "\n".join(_etc)
        
        # 【重要修正】スコープ外変数の直接参照を廃止し、引数で渡す仕様へ変更
        keyword = kks(name, skill, sk, str(i.get(2, '')), str(i.get(4, '')))

        products.append([
            sk_n, sk, code, name, time_val, unit, skill, abnormal, 
            grossWeight, wgt, spawn, cycle_val, standard, material, 
            raw, raw_mfr, one_box, pallet, tape, box, bag, etc, keyword
        ])

    products = [p for p in products if p[0] != ""]
    products.sort(key=lambda x: int(x[0].replace("SK", "").replace("ZZ", "")))

    for idx, row in enumerate(products, start=1):
        row.insert(0, idx)

    columns = [
        'num', 'slug', 'sk', 'code', 'name', 'time_val', 'unit', 'skill', 
        'abnormal', 'grossWeight', 'wgt', 'spawn', 'cycle_val', 'standard', 
        'material', 'raw', 'raw_mfr', 'one_box', 'pallet', 'tape', 'box', 
        'bag', 'etc', 'keyword'
    ]
    
    new_df = pl.DataFrame(products, schema=columns, orient="row")
    
    # 出力先ディレクトリが存在しない場合のエラーを防ぐ処理を追加
    output_dir = base_dir / 'csv'
    output_dir.mkdir(exist_ok=True)
    new_df.write_csv(output_dir / 'data.csv')

if __name__ == "__main__":
    main()

Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 7, falling back to string


In [2]:
import requests
def fetch_html(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

print(fetch_html('https://t-rance.net/backup/user_uuid'))
print(fetch_html('https://t-rance.net/backup/memo'))
print(fetch_html('https://t-rance.net/backup/stopcode'))
print(fetch_html('https://t-rance.net/backup/shift'))
print(fetch_html('https://t-rance.net/backup/dialogue'))

{"message":"Backup created successfully","file":"x_log/user_uuid_backup_202601250234.csv"}
{"message":"Backup created successfully","file":"x_log/memo_backup_202601250234.csv"}
{"message":"Backup created successfully","file":"x_log/stopcode_backup_202601250234.csv"}
{"message":"Backup created successfully","file":"x_log/shift_backup_202601250234.csv"}
{"message":"Backup created successfully","file":"x_log/dialogue_backup_202601250234.csv"}


In [4]:
%pip install pyautogui

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pyscreeze-1.0.1.tar.gz (27 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject